In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE SEMANTIC VIEW INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.INSURANCE_CLAIMS_SV

TABLES (
    fact_claims AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIMS
      PRIMARY KEY (CLAIM_KEY)
      WITH SYNONYMS = ('claims', 'insurance claims')
      COMMENT = 'Main fact table with claim financials, fraud indicators, and weather correlation',

    fact_expense AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.FACT_CLAIM_EXPENSE
      PRIMARY KEY (EXPENSE_KEY)
      WITH SYNONYMS = ('expenses', 'claim expenses')
      COMMENT = 'Expense tracking with budgeted vs actual variance',

    dim_date AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_DATE
      PRIMARY KEY (DATE_KEY)
      WITH SYNONYMS = ('calendar', 'dates')
      COMMENT = 'Date dimension with day, month, quarter, year grain',

    dim_geography AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_GEOGRAPHY
      PRIMARY KEY (GEOGRAPHY_KEY)
      WITH SYNONYMS = ('location', 'geography')
      COMMENT = 'Geographic dimension with state, city, zip, and region',

    dim_claim_type AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_CLAIM_TYPE
      PRIMARY KEY (CLAIM_TYPE_KEY)
      WITH SYNONYMS = ('line of business', 'LOB')
      COMMENT = 'Claim type dimension by line of business',

    dim_loss_cause AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_LOSS_CAUSE
      PRIMARY KEY (LOSS_CAUSE_KEY)
      WITH SYNONYMS = ('loss reason', 'cause of loss')
      COMMENT = 'Loss cause with weather-related categorization',

    dim_weather AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_WEATHER_EVENT
      PRIMARY KEY (WEATHER_EVENT_KEY)
      WITH SYNONYMS = ('weather', 'catastrophe')
      COMMENT = 'Weather event dimension from catastrophe data',

    dim_policy AS INSURANCE_CLAIM_DB.CONSUMPTION_LAYER.DIM_POLICY
      PRIMARY KEY (POLICY_KEY)
      WITH SYNONYMS = ('insurance policy', 'policy')
      COMMENT = 'Policy details with coverage and premium'
)

RELATIONSHIPS (
    claims_to_date AS fact_claims (DATE_KEY) REFERENCES dim_date,
    claims_to_geography AS fact_claims (GEOGRAPHY_KEY) REFERENCES dim_geography,
    claims_to_claim_type AS fact_claims (CLAIM_TYPE_KEY) REFERENCES dim_claim_type,
    claims_to_loss_cause AS fact_claims (LOSS_CAUSE_KEY) REFERENCES dim_loss_cause,
    claims_to_weather AS fact_claims (WEATHER_EVENT_KEY) REFERENCES dim_weather,
    claims_to_policy AS fact_claims (POLICY_KEY) REFERENCES dim_policy,
    expense_to_claims AS fact_expense (CLAIM_KEY) REFERENCES fact_claims
)

FACTS (
    fact_claims.fraud_flag AS CASE WHEN FRAUD_INDICATOR THEN 1 ELSE 0 END
      COMMENT = 'Numeric fraud flag (1=fraud, 0=clean)',
    fact_claims.weather_flag AS CASE WHEN IS_WEATHER_RELATED THEN 1 ELSE 0 END
      COMMENT = 'Numeric weather-related flag',
    fact_claims.cat_flag AS CASE WHEN CAT_INDICATOR THEN 1 ELSE 0 END
      COMMENT = 'Numeric catastrophe flag',
    fact_expense.budgeted_total AS BUDGETED_LEGAL_FEES + BUDGETED_ADJUSTOR_COSTS + BUDGETED_INVESTIGATION_CHARGES + BUDGETED_ULAE
      COMMENT = 'Total budgeted expense per claim'
)

DIMENSIONS (
    dim_date.calendar_year AS dim_date."YEAR"
      WITH SYNONYMS = ('year')
      COMMENT = 'Calendar year (2020-2026)',
    dim_date.calendar_quarter AS dim_date."QUARTER"
      WITH SYNONYMS = ('quarter')
      COMMENT = 'Calendar quarter (1-4)',
    dim_date.calendar_month AS dim_date."MONTH"
      WITH SYNONYMS = ('month number')
      COMMENT = 'Calendar month number (1-12)',
    dim_date.month_name_dim AS dim_date.MONTH_NAME
      WITH SYNONYMS = ('month name', 'month')
      COMMENT = 'Month name abbreviation',
    dim_date.year_month_period AS dim_date.YEAR_MONTH
      WITH SYNONYMS = ('year-month', 'monthly period')
      COMMENT = 'Year-month period (YYYY-MM)',
    dim_date.year_quarter_period AS dim_date.YEAR_QUARTER
      WITH SYNONYMS = ('year-quarter', 'quarterly period')
      COMMENT = 'Year-quarter period (YYYY-Q)',

    dim_geography.state_name AS dim_geography.STATE_NAME
      WITH SYNONYMS = ('state')
      COMMENT = 'Full state name',
    dim_geography.region AS dim_geography.REGION
      WITH SYNONYMS = ('geographic region', 'area')
      COMMENT = 'Geographic region (West, South, Southeast, Northeast, Midwest)',
    dim_geography.city AS dim_geography.CITY
      WITH SYNONYMS = ('city name')
      COMMENT = 'City name',

    dim_claim_type.claim_type AS dim_claim_type.CLAIM_TYPE
      WITH SYNONYMS = ('insurance type', 'line of business')
      COMMENT = 'Type of insurance claim / line of business',
    dim_claim_type.claim_category AS dim_claim_type.CLAIM_CATEGORY
      WITH SYNONYMS = ('claim group', 'business category')
      COMMENT = 'Claim category (Personal, Commercial, Specialty)',

    dim_loss_cause.loss_cause AS dim_loss_cause.LOSS_CAUSE
      WITH SYNONYMS = ('cause of loss', 'root cause')
      COMMENT = 'Root cause of the insurance claim',
    dim_loss_cause.loss_category AS dim_loss_cause.LOSS_CATEGORY
      WITH SYNONYMS = ('loss type', 'cause category')
      COMMENT = 'Loss cause category (Weather-Related, Accident, Crime, etc.)',
    dim_loss_cause.is_weather_related AS dim_loss_cause.IS_WEATHER_RELATED
      WITH SYNONYMS = ('weather related flag')
      COMMENT = 'Whether the loss cause is weather related (true/false)',
    dim_loss_cause.weather_condition_type AS dim_loss_cause.WEATHER_CONDITION_MAPPING
      WITH SYNONYMS = ('weather type')
      COMMENT = 'Mapped weather condition for weather-related losses',

    dim_weather.event_name AS dim_weather.EVENT_NAME
      WITH SYNONYMS = ('catastrophe name', 'weather event name')
      COMMENT = 'Name of the catastrophe/weather event',
    dim_weather.weather_condition AS dim_weather.WEATHER_CONDITION
      WITH SYNONYMS = ('event type', 'weather category')
      COMMENT = 'Type of weather event (Hurricane, Wildfire, Flood, etc.)',
    dim_weather.event_severity AS dim_weather.EVENT_SEVERITY_TIER
      WITH SYNONYMS = ('weather severity', 'event severity')
      COMMENT = 'Severity tier (Catastrophic, Severe, Major, Moderate)',
    dim_weather.primary_weather_driver_dim AS dim_weather.PRIMARY_WEATHER_DRIVER
      WITH SYNONYMS = ('weather driver')
      COMMENT = 'Primary weather driver (Wind/Rain, Wind/Hail, etc.)',

    dim_policy.coverage_type AS dim_policy.COVERAGE_TYPE
      WITH SYNONYMS = ('coverage', 'policy coverage')
      COMMENT = 'Type of policy coverage',
    dim_policy.policy_status AS dim_policy.POLICY_STATUS
      WITH SYNONYMS = ('policy state')
      COMMENT = 'Current policy status',

    fact_claims.claim_status AS fact_claims.CLAIM_STATUS
      WITH SYNONYMS = ('status')
      COMMENT = 'Claim status (Open, Closed, Approved, Pending Review, Rejected, Stalled)',
    fact_claims.claim_severity AS fact_claims.CLAIM_SEVERITY
      WITH SYNONYMS = ('severity', 'severity level')
      COMMENT = 'Claim severity (Minor, Moderate, Significant, Severe, Catastrophic)',
    fact_claims.fraud_risk_tier AS fact_claims.FRAUD_RISK_TIER
      WITH SYNONYMS = ('fraud tier', 'fraud risk')
      COMMENT = 'Fraud risk classification (High Risk, Medium Risk, No Flag)',
    fact_claims.payment_type AS fact_claims.PAYMENT_TYPE
      WITH SYNONYMS = ('payment method')
      COMMENT = 'Type of claim payment',
    fact_expense.expense_category AS fact_expense.EXPENSE_CATEGORY
      WITH SYNONYMS = ('expense type')
      COMMENT = 'Category of the expense'
)

METRICS (
    fact_claims.total_revenue AS SUM(PAID_AMOUNT)
      WITH SYNONYMS = ('total paid', 'revenue', 'paid amount')
      COMMENT = 'Total paid claim amount - primary revenue metric',
    fact_claims.total_incurred_loss AS SUM(INCURRED_LOSS)
      WITH SYNONYMS = ('total incurred', 'incurred loss')
      COMMENT = 'Total incurred loss amount',
    fact_claims.reserve_fund AS SUM(RESERVE_AMOUNT)
      WITH SYNONYMS = ('total reserves', 'reserve', 'reserves')
      COMMENT = 'Total reserve fund set aside for outstanding claims',
    fact_claims.total_net_incurred AS SUM(NET_INCURRED)
      WITH SYNONYMS = ('net loss', 'net incurred')
      COMMENT = 'Net incurred loss after all recoveries',
    fact_claims.total_recovery AS SUM(RECOVERY_AMOUNT)
      WITH SYNONYMS = ('recoveries', 'total recovered')
      COMMENT = 'Total recovery amount',
    fact_claims.claim_count AS COUNT(CLAIM_KEY)
      WITH SYNONYMS = ('number of claims', 'claims count')
      COMMENT = 'Total count of insurance claims',
    fact_claims.avg_claim_amount AS AVG(PAID_AMOUNT)
      WITH SYNONYMS = ('average claim', 'average paid')
      COMMENT = 'Average paid amount per claim',
    fact_claims.fraud_claim_count AS SUM(fact_claims.fraud_flag)
      WITH SYNONYMS = ('fraudulent claims', 'fraud count')
      COMMENT = 'Count of fraudulent claims',
    fact_claims.weather_claim_count AS SUM(fact_claims.weather_flag)
      WITH SYNONYMS = ('weather claims count')
      COMMENT = 'Count of weather-related claims',
    fact_claims.catastrophe_claim_count AS SUM(fact_claims.cat_flag)
      WITH SYNONYMS = ('cat claims count')
      COMMENT = 'Count of catastrophe-linked claims',
    fact_claims.avg_fraud_score AS AVG(FRAUD_SCORE)
      WITH SYNONYMS = ('average fraud score')
      COMMENT = 'Average fraud risk score',
    fact_claims.avg_days_to_close AS AVG(DAYS_TO_CLOSE)
      WITH SYNONYMS = ('average close time', 'settlement time')
      COMMENT = 'Average days from open to close',
    fact_expense.total_expenses AS SUM(TOTAL_ACTUAL_EXPENSE)
      WITH SYNONYMS = ('total actual expenses', 'actual expenses')
      COMMENT = 'Total actual claim expenses (legal, adjustor, investigation, ULAE)',
    fact_expense.total_budget_variance AS SUM(TOTAL_EXPENSE_VARIANCE)
      WITH SYNONYMS = ('budget variance', 'expense variance')
      COMMENT = 'Total variance between actual and budgeted expenses',
    fact_expense.total_budgeted_expense AS SUM(fact_expense.budgeted_total)
      WITH SYNONYMS = ('total budget', 'budgeted expenses')
      COMMENT = 'Total budgeted expense amount',
    fact_expense.avg_expense_per_claim AS AVG(TOTAL_ACTUAL_EXPENSE)
      WITH SYNONYMS = ('average expense')
      COMMENT = 'Average actual expense per claim'
)

COMMENT = 'Insurance claims analytics semantic view for revenue, expense, loss cause, and fraud analytics'
AI_SQL_GENERATION 'Round all monetary values to 2 decimal places. For fraud analysis use fraud_risk_tier dimension and avg_fraud_score metric. For weather impact use loss_category and weather_condition dimensions. When no date filter is specified, include all available dates.'